# Visualization Gallery — Every Chart Type I Could Think Of

This notebook generates a large gallery of common (and some less-common)
data-visualization types. For each chart there is:

1. A **short description** (what it shows, when to use it, pitfalls), then
2. A **code cell** that builds the chart from synthetic data and **saves a PNG**
   into `./images/`.

At the end, all rendered images are zipped into `./visualizations_gallery.zip`
so you can download the full set in one go.

**Libraries used:** matplotlib, seaborn, plotly (static export via kaleido),
networkx, scikit-learn, scipy, numpy, pandas.

Run cells top to bottom. The first code cell sets up shared imports and the
output directory.

In [1]:
# Setup: imports + output directory
import os, math, itertools, warnings, shutil, zipfile, textwrap
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless-safe; figures are saved to disk
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import cm
from matplotlib.patches import Patch
import seaborn as sns
import networkx as nx
from scipy import stats
from scipy import sparse
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.decomposition import PCA
from sklearn.manifold import MDS
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import confusion_matrix
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

# Consistent style
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 120
plt.rcParams["savefig.bbox"] = "tight"

IMG = "./images"
os.makedirs(IMG, exist_ok=True)

RNG = np.random.default_rng(42)

def save(fig, name):
    """Save a matplotlib figure to ./images/<name>.png and close it."""
    path = os.path.join(IMG, name + ".png")
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print("saved", path)
    return path

def save_plotly(fig, name):
    path = os.path.join(IMG, name + ".png")
    fig.write_image(path, width=900, height=600, scale=2)
    print("saved", path)
    return path

print("Environment ready. Images will be written to:", os.path.abspath(IMG))

Environment ready. Images will be written to: P:\laney\images


## Line Chart

The most basic chart: a series of points connected by straight segments. Best for showing how a quantity changes **over a continuous variable** (time, distance, temperature). Pitfall: a line implies a meaningful ordering and continuity — don't use it for categorical data.

In [2]:
t = np.linspace(0, 4*np.pi, 200)
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(t, np.sin(t), label="sin(t)")
ax.plot(t, np.cos(t), label="cos(t)", linestyle="--")
ax.set(title="Line Chart — sin and cos", xlabel="t", ylabel="amplitude")
ax.legend()
save(fig, "01_line_chart")

saved ./images\01_line_chart.png


'./images\\01_line_chart.png'

## Multi-line Time Series

Several lines on shared axes — the classic way to compare trends across groups over time. Watch out for **too many lines** (spaghetti chart); use direct labels or small multiples instead when n > ~5.

In [3]:
dates = pd.date_range("2024-01-01", periods=90, freq="D")
df = pd.DataFrame({
    "A": RNG.normal(0,1,90).cumsum(),
    "B": RNG.normal(0,1,90).cumsum()+5,
    "C": RNG.normal(0,1,90).cumsum()+10,
}, index=dates)
fig, ax = plt.subplots(figsize=(9,5))
df.plot(ax=ax, linewidth=2)
ax.set(title="Multi-line Time Series (90 days)", xlabel="date", ylabel="value")
ax.legend(title="series")
save(fig, "02_multiline_timeseries")

saved ./images\02_multiline_timeseries.png


'./images\\02_multiline_timeseries.png'

## Area Chart

Like a line chart but the area below the line is filled. Emphasizes **cumulative magnitude** over a range. A *stacked* area chart shows parts of a whole; an *overlaid* area chart just emphasizes magnitude. Beware: stacked areas hide the shape of upper layers.

In [4]:
x = np.linspace(0, 10, 100)
y1, y2 = np.sin(x)+2, np.cos(x)+2
fig, ax = plt.subplots(figsize=(8,5))
ax.fill_between(x, 0, y1, alpha=0.5, label="sin+2")
ax.fill_between(x, 0, y2, alpha=0.5, label="cos+2")
ax.set(title="Area Chart", xlabel="x", ylabel="y"); ax.legend()
save(fig, "03_area_chart")

saved ./images\03_area_chart.png


'./images\\03_area_chart.png'

## Stacked Area Chart

Multiple quantitative series stacked on top of each other — the total height is the sum. Great for **composition over time** when the total matters as much as the components. Pitfall: comparing non-adjacent layers is hard because their baselines move.

In [5]:
x = np.arange(0, 12)
A = RNG.integers(10, 30, 12)
B = RNG.integers(5, 25, 12)
C = RNG.integers(2, 15, 12)
fig, ax = plt.subplots(figsize=(8,5))
ax.stackplot(x, A, B, C, labels=["A","B","C"], alpha=0.8)
ax.set(title="Stacked Area Chart", xlabel="month", ylabel="units"); ax.legend()
save(fig, "04_stacked_area")

saved ./images\04_stacked_area.png


'./images\\04_stacked_area.png'

## Bar Chart (vertical)

Categorical quantities shown as rectangular bars — height encodes value. Best for comparing **discrete categories**. Pitfall: starting the y-axis somewhere other than zero misleads the eye about relative magnitude.

In [6]:
cats = list("ABCDEFG")
vals = RNG.integers(5, 40, len(cats))
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(cats, vals, color="steelblue")
ax.set(title="Bar Chart", xlabel="category", ylabel="value")
save(fig, "05_bar_chart")

saved ./images\05_bar_chart.png


'./images\\05_bar_chart.png'

## Grouped Bar Chart

Two-or-more series per category, side by side. Compares values **within**
    a category and across categories. Pitfall: becomes unreadable with more
    than ~3 series or 8 categories — switch to stacked or small multiples.

In [7]:
cats = ["Q1","Q2","Q3","Q4"]
x = np.arange(len(cats)); w = 0.35
g1 = RNG.integers(10,40,4); g2 = RNG.integers(10,40,4)
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(x-w/2, g1, w, label="2023")
ax.bar(x+w/2, g2, w, label="2024")
ax.set_xticks(x); ax.set_xticklabels(cats); ax.legend()
ax.set(title="Grouped Bar Chart", xlabel="quarter", ylabel="revenue")
save(fig, "06_grouped_bar")

saved ./images\06_grouped_bar.png


'./images\\06_grouped_bar.png'

## Stacked Bar Chart

Segments stacked within a single bar per category — shows the **parts of a whole** across categories. Pitfall: middle segments are hard to compare because their baselines float.

In [8]:
cats = ["Q1","Q2","Q3","Q4"]
s1 = RNG.integers(5,20,4); s2 = RNG.integers(5,20,4); s3 = RNG.integers(5,20,4)
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(cats, s1, label="Hardware")
ax.bar(cats, s2, bottom=s1, label="Software")
ax.bar(cats, s3, bottom=s1+s2, label="Services")
ax.set(title="Stacked Bar Chart", ylabel="revenue"); ax.legend()
save(fig, "07_stacked_bar")

saved ./images\07_stacked_bar.png


'./images\\07_stacked_bar.png'

## Horizontal Bar Chart

Bars run left-to-right. Use it when **category labels are long** or when ranking from largest to smallest — the eye is better at comparing horizontal lengths than vertical heights.

In [9]:
items = ["Solar flux variation","Magnetic field reconnection","CME propagation",
        "Particle acceleration","Heliospheric current sheet","Interplanetary shock"]
vals = RNG.integers(5, 50, len(items))
order = np.argsort(vals)
fig, ax = plt.subplots(figsize=(9,5))
ax.barh([items[i] for i in order], vals[order], color="darkorange")
ax.set(title="Horizontal Bar Chart", xlabel="count")
save(fig, "08_horizontal_bar")

saved ./images\08_horizontal_bar.png


'./images\\08_horizontal_bar.png'

## Histogram

Splits a continuous variable into bins and counts observations per bin. Shows the **shape of a distribution** (mode, skew, tails). Pitfall: bin width dramatically changes the story — try several.

In [10]:
data = RNG.normal(0,1,2000)
fig, ax = plt.subplots(figsize=(8,5))
ax.hist(data, bins=30, color="slateblue", edgecolor="white")
ax.set(title="Histogram", xlabel="value", ylabel="count")
save(fig, "09_histogram")

saved ./images\09_histogram.png


'./images\\09_histogram.png'

## Density Plot (KDE)

A smoothed estimate of the probability density. Reads like a continuous histogram without binning artefacts. Pitfall: KDE can suggest density where there are no data points (especially near boundaries).

In [11]:
a = RNG.normal(0,1,500); b = RNG.normal(3,1.2,500)
fig, ax = plt.subplots(figsize=(8,5))
sns.kdeplot(a, fill=True, alpha=0.4, ax=ax, label="A")
sns.kdeplot(b, fill=True, alpha=0.4, ax=ax, label="B")
ax.set(title="Density Plot (KDE)"); ax.legend()
save(fig, "10_density_kde")

saved ./images\10_density_kde.png


'./images\\10_density_kde.png'

## Box Plot

Summary of a distribution: median, quartiles, whiskers, and outliers. Great for comparing **distributions across groups** compactly. Pitfall: hides the underlying shape — a violin or swarm overlay helps.

In [12]:
groups = [RNG.normal(0,1,200), RNG.normal(2,1.5,200),
            RNG.normal(-1,2,200), RNG.normal(1,0.8,200)]
fig, ax = plt.subplots(figsize=(8,5))
ax.boxplot(groups, labels=["A","B","C","D"])
ax.set(title="Box Plot", ylabel="value")
save(fig, "11_box_plot")

saved ./images\11_box_plot.png


'./images\\11_box_plot.png'

## Violin Plot

Box plot + KDE: shows the full distribution shape (mirrored). Better than a box for **bimodal or skewed** data. Pitfall: still just an estimate — strip the data points for small samples.

In [13]:
df = pd.DataFrame({
    "val": np.concatenate([RNG.normal(0,1,300), RNG.normal(3,1.5,300),
                           RNG.normal(6,1,300)]),
    "grp": ["A"]*300 + ["B"]*300 + ["C"]*300})
fig, ax = plt.subplots(figsize=(8,5))
sns.violinplot(data=df, x="grp", y="val", ax=ax, inner="quartile")
ax.set(title="Violin Plot")
save(fig, "12_violin_plot")

saved ./images\12_violin_plot.png


'./images\\12_violin_plot.png'

## Strip / Swarm Plot

Each observation drawn as a point, jittered so they don't overlap (swarm). Best for **small-to-moderate samples** where you want to see every observation, not a summary.

In [14]:
df = pd.DataFrame({
    "val": np.concatenate([RNG.normal(0,1,80), RNG.normal(3,1.2,80)]),
    "grp": ["A"]*80 + ["B"]*80})
fig, ax = plt.subplots(figsize=(8,5))
sns.swarmplot(data=df, x="grp", y="val", size=4, ax=ax)
ax.set(title="Swarm Plot")
save(fig, "13_swarm_plot")

saved ./images\13_swarm_plot.png


'./images\\13_swarm_plot.png'

## Scatter Plot

Two variables, one point per observation. The canonical way to inspect **relationship between two continuous variables**. Pitfall: overplotting with many points — use hex/2D-density or alpha blending.

In [15]:
n=300
x = RNG.normal(0,1,n); y = 2*x + RNG.normal(0,0.8,n)
fig, ax = plt.subplots(figsize=(8,6))
ax.scatter(x, y, alpha=0.6, edgecolor="none")
ax.set(title="Scatter Plot", xlabel="x", ylabel="y")
save(fig, "14_scatter")

saved ./images\14_scatter.png


'./images\\14_scatter.png'

## Bubble Chart (colored & sized scatter)

Adds a third (color) and fourth (size) variable to a scatter. Lets you show **4 dimensions** on a 2D plot. Pitfall: more than four quickly becomes unreadable — try faceting instead.

In [16]:
n=150
x=RNG.normal(0,1,n); y=RNG.normal(0,1,n)
z=RNG.uniform(0,1,n); s=RNG.uniform(20,400,n)
fig, ax = plt.subplots(figsize=(8,6))
sc = ax.scatter(x, y, c=z, s=s, alpha=0.6, cmap="viridis", edgecolor="none")
plt.colorbar(sc, ax=ax, label="z"); ax.set(title="Bubble Chart")
save(fig, "15_bubble_chart")

saved ./images\15_bubble_chart.png


'./images\\15_bubble_chart.png'

## Hexbin / 2-D Density

For thousands of points: bins the plane into hexagons and colours each by count. Reveals **dense regions** that a scatter would smear into one black blob. Pitfall: hex size matters — try a few.

In [17]:
n=20000
x=RNG.normal(0,1,n); y=x*0.5 + RNG.normal(0,1,n)
fig, ax = plt.subplots(figsize=(8,6))
hb = ax.hexbin(x, y, gridsize=40, cmap="magma", mincnt=1)
plt.colorbar(hb, ax=ax, label="count"); ax.set(title="Hexbin (2D density)")
save(fig, "16_hexbin")

saved ./images\16_hexbin.png


'./images\\16_hexbin.png'

## 2-D KDE Contour

Smoothed contours of bivariate density — read it like a topographic map of where the data lives. Pitfall: tails are extrapolated; check the data range against contour extent.

In [18]:
n=2000
x=RNG.normal(0,1,n); y=0.7*x + RNG.normal(0,1,n)
fig, ax = plt.subplots(figsize=(8,6))
sns.kdeplot(x=x, y=y, fill=True, levels=12, cmap="crest", ax=ax)
ax.set(title="2D KDE Contour")
save(fig, "17_kde2d")

saved ./images\17_kde2d.png


'./images\\17_kde2d.png'

## Pie Chart

Slices of a circle showing parts of a whole. Use only for **a few categories whose total is meaningful**. Pitfall: humans are bad at comparing angles — a bar chart is almost always clearer.

In [19]:
vals = RNG.integers(10,40,5)
labels = ["A","B","C","D","E"]
fig, ax = plt.subplots(figsize=(7,7))
ax.pie(vals, labels=labels, autopct='%1.0f%%', startangle=90,
       wedgeprops=dict(width=0.4))
ax.set(title="Pie Chart")
save(fig, "18_pie_chart")

saved ./images\18_pie_chart.png


'./images\\18_pie_chart.png'

## Donut Chart

A pie chart with the centre cut out. The hole can hold a label. Same caveats as a pie; slightly easier to compare arcs because the eye compares lengths along the ring, not full wedge areas.

In [20]:
vals = RNG.integers(15,35,4)
labels = ["Alpha","Beta","Gamma","Delta"]
fig, ax = plt.subplots(figsize=(7,7))
ax.pie(vals, labels=labels, autopct='%1.0f%%', startangle=90,
       wedgeprops=dict(width=0.35), pctdistance=0.78)
ax.set(title="Donut Chart")
save(fig, "19_donut")

saved ./images\19_donut.png


'./images\\19_donut.png'

## Heatmap

A 2-D grid coloured by value — shows a matrix at a glance. Common for **correlations, confusion matrices, gene expression**. Pitfall: the colour map choice (e.g. jet) can invent structure; prefer perceptually uniform maps like viridis.

In [21]:
corr = pd.DataFrame(RNG.uniform(-1,1,(8,8)))
corr = (corr + corr.T)/2
arr = np.array(corr.values, copy=True); np.fill_diagonal(arr, 1); corr = pd.DataFrame(arr)
fig, ax = plt.subplots(figsize=(7,6))
sns.heatmap(corr, annot=True, fmt=".1f", cmap="coolwarm", center=0, ax=ax,
            xticklabels=list("ABCDEFGH"), yticklabels=list("ABCDEFGH"))
ax.set(title="Heatmap (correlation-style)")
save(fig, "20_heatmap")

saved ./images\20_heatmap.png


'./images\\20_heatmap.png'

## Correlation Heatmap

Special heatmap showing the **Pearson correlation** between every pair of numeric columns. A fast diagnostic for redundancy and multicollinearity.

In [22]:
df = pd.DataFrame(RNG.normal(0,1,(200,6)),
              columns=["speed","rpm","temp","vibe","load","fuel"])
df["rpm"] = df["speed"]*0.8 + RNG.normal(0,0.3,200)
df["fuel"] = -df["speed"]*0.5 + RNG.normal(0,0.5,200)
fig, ax = plt.subplots(figsize=(7,6))
sns.heatmap(df.corr(), annot=True, cmap="vlag", center=0, ax=ax)
ax.set(title="Correlation Heatmap")
save(fig, "21_correlation_heatmap")

saved ./images\21_correlation_heatmap.png


'./images\\21_correlation_heatmap.png'

## Pair Plot (Scatter Matrix)

Pairwise scatter of every numeric column + marginal density on the diagonal. The fastest way to **see all pairwise relationships** in a dataframe. Pitfall: O(p²) panels — doesn't scale past ~10 columns.

In [23]:
df = pd.DataFrame(RNG.normal(0,1,(200,4)), columns=list("ABCD"))
df["grp"] = RNG.choice(["x","y","z"], 200)
g = sns.pairplot(df, hue="grp", corner=True, height=1.8)
g.figure.suptitle("Pair Plot", y=1.02)
fig = g.figure
save(fig, "22_pairplot")

saved ./images\22_pairplot.png


'./images\\22_pairplot.png'

## Joint Plot (scatter + marginal)

A scatter with **histograms or KDEs on the margins** for each axis. Great for showing the bivariate relationship *and* each marginal distribution at once.

In [24]:
n=300
df = pd.DataFrame({"x":RNG.normal(0,1,n), "y":RNG.normal(1,1.3,n)})
df["y"] += 0.6*df["x"]
g = sns.jointplot(data=df, x="x", y="y", kind="hex", height=6)
g.figure.suptitle("Joint Plot (hex)", y=1.02)
fig = g.figure
save(fig, "23_jointplot")

saved ./images\23_jointplot.png


'./images\\23_jointplot.png'

## Line with Error Bars

A line chart plus an indication of uncertainty (std, SEM, or CI) at each point. Essential when summarizing **replicated measurements** so the reader knows the precision.

In [25]:
x = np.arange(0,10); y = np.sin(x)
err = RNG.uniform(0.1,0.5,10)
fig, ax = plt.subplots(figsize=(8,5))
ax.errorbar(x, y, yerr=err, fmt="-o", capsize=4, color="darkred")
ax.set(title="Line with Error Bars", xlabel="x", ylabel="sin(x) ± err")
save(fig, "24_errorbar")

saved ./images\24_errorbar.png


'./images\\24_errorbar.png'

## Boxen (Letter-Value) Plot

A box plot that extends into the tails with more quantiles. Better than a box for **heavy-tailed or very large-sample** distributions where the whiskers hide too much.

In [26]:
df = pd.DataFrame({"v": np.concatenate(
        [RNG.normal(0,1,300), RNG.normal(8,2,300)]),
        "g": ["A"]*300 + ["B"]*300})
fig, ax = plt.subplots(figsize=(8,5))
sns.boxenplot(data=df, x="g", y="v", ax=ax)
ax.set(title="Boxen (Letter-Value) Plot")
save(fig, "25_boxen")

saved ./images\25_boxen.png


'./images\\25_boxen.png'

## Ridge (Joy) Plot

Overlapping KDEs stacked vertically — looks like a mountain range. Excellent for **comparing many distributions** that share an x-axis. Pitfall: can be actively misleading if a peak's tail is clipped by the row above.

In [27]:
n = 1500; k = 7
df = pd.DataFrame({
    "v": RNG.normal(0,1,n),
    "g": np.tile(np.arange(k), n//k+1)[:n]})
df["v"] += df["g"].values*0.5
fig, ax = plt.subplots(figsize=(8,6))
sns.kdeplot(data=df, x="v", hue="g", fill=True, linewidth=0,
            common_norm=False, alpha=0.7, ax=ax)
ax.set(title="Ridge-style Plot (overlapping KDEs by group)")
save(fig, "26_ridge")

saved ./images\26_ridge.png


'./images\\26_ridge.png'

## Stem (Lollipop) Plot

Vertical lines topped by a dot — like a bar chart but lighter. Nice for **discrete sequences** (e.g., a spectrum, impulse response, or counts with few categories) where the bar's solid fill would be too heavy.

In [28]:
x = np.arange(-10,11)
y = np.sinc(x/3)
fig, ax = plt.subplots(figsize=(9,5))
ax.stem(x, y, linefmt="steelblue", markerfmt="o", basefmt="k")
ax.set(title="Stem (Lollipop) Plot", xlabel="index", ylabel="value")
save(fig, "27_stem")

saved ./images\27_stem.png


'./images\\27_stem.png'

## Step (Staircase) Plot

A line drawn as a staircase. Use for **piecewise-constant** data (e.g., square waves, digital signals, cumulative counts at integer steps) where a sloped line would misrepresent the value between steps.

In [29]:
x = np.arange(0,10); y = RNG.integers(1,8,10)
fig, ax = plt.subplots(figsize=(8,5))
ax.step(x, y, where="mid", color="purple")
ax.fill_between(x, 0, y, step="mid", alpha=0.2, color="purple")
ax.set(title="Step / Staircase Plot")
save(fig, "28_step")

saved ./images\28_step.png


'./images\\28_step.png'

## Candlestick (OHLC)

Open–High–Low–Close bars per time step, green = up day, red = down day. The canonical **financial** chart for price history — encodes four numbers per candle.

In [30]:
days = 40
o = 100 + RNG.normal(0,0.5,days).cumsum()
c = o + RNG.normal(0,0.6,days)
h = np.maximum(o,c) + np.abs(RNG.normal(0,0.3,days))
l = np.minimum(o,c) - np.abs(RNG.normal(0,0.3,days))
fig, ax = plt.subplots(figsize=(9,6))
for i,(O,H,L,C) in enumerate(zip(o,h,l,c)):
    color = "seagreen" if C>=O else "crimson"
    ax.plot([i,i],[L,H], color=color)
    ax.add_patch(plt.Rectangle((i-0.3,min(O,C)), 0.6, abs(C-O),
                              color=color))
ax.set(title="Candlestick (OHLC)", xlabel="day", ylabel="price")
save(fig, "29_candlestick")

saved ./images\29_candlestick.png


'./images\\29_candlestick.png'

## Radar / Spider Chart

Each quantitative variable becomes an axis radiating from the centre, values joined into a polygon. Useful for **comparing a few entities on the same set of metrics**. Pitfall: ordering of axes and radial scaling both change the shape — read carefully.

In [31]:
labels=["Speed","Reliability","Cost","Safety","Comfort","Range"]
n=len(labels)
v1=RNG.uniform(2,5,n); v2=RNG.uniform(2,5,n)
angles=np.linspace(0,2*np.pi,n,endpoint=False).tolist(); angles += angles[:1]
v1=list(v1)+[v1[0]]; v2=list(v2)+[v2[0]]
fig, ax = plt.subplots(figsize=(7,7), subplot_kw=dict(polar=True))
ax.plot(angles, v1, label="Model A"); ax.fill(angles, v1, alpha=0.2)
ax.plot(angles, v2, label="Model B"); ax.fill(angles, v2, alpha=0.2)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels)
ax.set(title="Radar / Spider Chart"); ax.legend(loc="upper right")
save(fig, "30_radar")

saved ./images\30_radar.png


'./images\\30_radar.png'

## Polar Bar Chart

Bars arranged around a circle instead of on a line. Useful for **cyclic quantities** — wind direction, time-of-day activity, compass bearings. The wrap-around is natural here; it isn't on a linear bar.

In [32]:
N=24; theta=np.linspace(0,2*np.pi,N,endpoint=False)
width=2*np.pi/N*0.9
radii=RNG.uniform(1,10,N)
fig, ax = plt.subplots(figsize=(7,7), subplot_kw=dict(projection="polar"))
ax.bar(theta, radii, width=width, color=plt.cm.twilight(theta/2/np.pi))
ax.set_title("Polar Bar Chart (24 sectors)")
save(fig, "31_polar_bar")

saved

 ./images\31_polar_bar.png


'./images\\31_polar_bar.png'

## Polar Line (spiral)

A line drawn in polar coordinates. Great for showing how an **angle-resolved signal** evolves — e.g., antenna gain pattern, directional response — or for the aesthetics of a parameter swept in angle.

In [33]:
theta=np.linspace(0,8*np.pi,500)
r=theta/2
fig, ax = plt.subplots(figsize=(7,7), subplot_kw=dict(projection="polar"))
ax.plot(theta, r, color="teal"); ax.set_title("Polar Line (spiral)")
save(fig, "32_polar_line")

saved ./images\32_polar_line.png


'./images\\32_polar_line.png'

## Contour Plot

Lines of equal value of a 2-D scalar field. Familiar from weather maps and topology. Great for **f(x, y)** when the function is smooth.

In [34]:
x=np.linspace(-3,3,200); y=np.linspace(-3,3,200)
X,Y=np.meshgrid(x,y); Z=np.sin(X)*np.cos(Y)*np.exp(-(X**2+Y**2)/6)
fig, ax = plt.subplots(figsize=(7,6))
cs=ax.contourf(X,Y,Z, levels=20, cmap="viridis")
ax.contour(X,Y,Z, levels=20, colors="k", linewidths=0.3)
plt.colorbar(cs, ax=ax); ax.set(title="Contour Plot")
save(fig, "33_contour")

saved ./images\33_contour.png


'./images\\33_contour.png'

## Quiver (Vector Field)

A grid of arrows showing the direction and magnitude of a vector field at each point. Use for **flow, gradient fields, or wind maps**. Keep the arrow sample density low or you'll get a black blob.

In [35]:
x=np.linspace(-2,2,18); y=np.linspace(-2,2,18)
X,Y=np.meshgrid(x,y)
U=-Y; V=X
fig, ax = plt.subplots(figsize=(7,6))
ax.quiver(X,Y,U,V, color="navy")
ax.set(title="Quiver (Vector Field)", aspect="equal")
save(fig, "34_quiver")

saved ./images\34_quiver.png


'./images\\34_quiver.png'

## Stream Plot

Like a quiver, but arrows become continuous **streamlines**. Reads much better than a quiver for fluid/flow visualization because the eye follows the curves naturally.

In [36]:
x=np.linspace(-2,2,150); y=np.linspace(-2,2,150)
X,Y=np.meshgrid(x,y); U=-Y; V=X
fig, ax = plt.subplots(figsize=(7,6))
ax.streamplot(x,y,U,V, density=1.2, color=np.hypot(U,V), cmap="cool")
ax.set(title="Stream Plot"); ax.set_aspect("equal")
save(fig, "35_streamplot")

saved ./images\35_streamplot.png


'./images\\35_streamplot.png'

## 3-D Surface

A 2-D function rendered as a height field in 3-D. Pretty, but **only use when the third dimension truly fits** — perspective can hide peaks. Often a contour is more legible.

In [37]:
x=np.linspace(-3,3,80); y=np.linspace(-3,3,80)
X,Y=np.meshgrid(x,y); Z=np.sin(X)*np.cos(Y)*np.exp(-(X**2+Y**2)/10)
fig=plt.figure(figsize=(8,6))
ax=fig.add_subplot(111, projection="3d")
ax.plot_surface(X,Y,Z, cmap="viridis", edgecolor="none", alpha=0.9)
ax.set_title("3D Surface")
save(fig, "36_surface3d")

saved ./images\36_surface3d.png


'./images\\36_surface3d.png'

## 3-D Wireframe

A surface rendered as a mesh of lines instead of a coloured sheet. Useful when you want to **see through** gaps in the field — e.g., showing one surface outlined over another.

In [38]:
x=np.linspace(-3,3,40); y=np.linspace(-3,3,40)
X,Y=np.meshgrid(x,y); Z=np.sin(np.sqrt(X**2+Y**2))
fig=plt.figure(figsize=(8,6))
ax=fig.add_subplot(111, projection="3d")
ax.plot_wireframe(X,Y,Z, color="darkgreen", linewidth=0.6)
ax.set_title("3D Wireframe")
save(fig, "37_wireframe")

saved ./images\37_wireframe.png


'./images\\37_wireframe.png'

## 3-D Scatter

Three continuous variables, one point per observation, in 3-D. Use to look for **cluster structure** in 3-D. Pitfall: a static 3-D plot on paper is ambiguous — rotate interactively or fall back to PCA + pairplot.

In [39]:
X,_=make_blobs(n_samples=300, centers=4, n_features=3, random_state=0)
fig=plt.figure(figsize=(8,6))
ax=fig.add_subplot(111, projection="3d")
ax.scatter(X[:,0],X[:,1],X[:,2], c=X[:,0], cmap="tab10", s=25, depthshade=True)
ax.set(title="3D Scatter (clusters)")
save(fig, "38_scatter3d")

saved ./images\38_scatter3d.png


'./images\\38_scatter3d.png'

## Treemap (squarify)

Hierarchical quantities shown as rectangles whose **area is proportional to value**. Good for parts-of-a-whole with many categories (where a pie would explode). Note: this chart uses matplotlib patches — no extra library needed beyond numpy.

In [40]:
vals=np.sort(RNG.integers(1,40,12))[::-1]
labels=[f"item {i+1}" for i in range(12)]
# squarified-flavoured layout with a minimal owner implementation
fig, ax = plt.subplots(figsize=(9,6))
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_xticks([]); ax.set_yticks([])
total=vals.sum()
yy=1.0
colors=plt.cm.tab20(np.linspace(0,1,len(vals)))
for v,c in zip(sorted(vals,reverse=True), colors[np.argsort(np.argsort(-vals))]):
    h=v/total
    ax.add_patch(plt.Rectangle((0, yy-h), 1, h, color=c, alpha=0.85))
    ax.text(0.5, yy-h/2, str(v), ha="center", va="center", color="white", fontweight="bold")
    yy-=h
ax.set(title="Treemap (horizontal strips, area = value)")
save(fig, "39_treemap")

saved ./images\39_treemap.png


'./images\\39_treemap.png'

## Waffle Chart

A grid of small squares, each representing one unit, coloured by category. A more honest alternative to a pie for **parts of a whole** because counting squares is easier than comparing angles.

In [41]:
vals=[35,28,22,15]; labels=["A","B","C","D"]
colors=["#4C72B0","#DD8452","#55A868","#C44E52"]
cols=10; rows=10; total=sum(vals)  # 100 cells
# build a 10x10 grid filled by category index, row by row
G=np.zeros((rows,cols), dtype=int)
cells_per_cat=[int(round(v/total*rows*cols)) for v in vals]
# fix rounding if needed so all cells are filled
cells_per_cat[-1]=rows*cols-sum(cells_per_cat[:-1])
idx=0
for ci,n_cells in enumerate(cells_per_cat):
    for _ in range(n_cells):
        r,cx=divmod(idx,cols); G[r,cx]=ci+1; idx+=1
fig, ax = plt.subplots(figsize=(7,7))
for r in range(rows):
    for cxx in range(cols):
        cat=int(G[r,cx])-1
        face = colors[cat] if cat>=0 else "whitesmoke"
        ax.add_patch(plt.Rectangle((cxx*1.05, (rows-1-r)*1.05), 1, 1,
                     facecolor=face, edgecolor="white"))
ax.set_xlim(-0.5,(cols-1)*1.05+1.5); ax.set_ylim(-0.5,(rows-1)*1.05+1.5)
ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=c,label=l) for c,l in zip(colors,labels)],
          loc="lower center", ncol=4, bbox_to_anchor=(0.5,-0.05))
ax.set(title="Waffle Chart (10x10, one square = 1% of total)")
save(fig, "40_waffle")

saved ./images\40_waffle.png


'./images\\40_waffle.png'

## Waterfall Chart

Series of bars where each starts where the previous ended — shows how increments build up to a total. Common in **finance** for revenue bridges and P&L breakdowns.

In [42]:
import matplotlib.patches as mpatches
steps=["Start","+Sales","+Subscriptions","-Refunds","-OpEx","Net"]
vals=[100,40,25,-8,-30, None]
base=0; fig,ax=plt.subplots(figsize=(9,5))
for i,(s,v) in enumerate(zip(steps,vals)):
    if v is None:  # total bar
        v=base
        ax.bar(i, v, color="slateblue", label="total")
        ax.text(i, v+2, f"{v}", ha="center")
        continue
    color="seagreen" if v>=0 else "crimson"
    ax.bar(i, v, bottom=base if v>=0 else base+v, color=color)
    ax.text(i, base + v + 1.5*(1 if v>=0 else 1), f"{v:+d}", ha="center")
    base += v
ax.set_xticks(range(len(steps))); ax.set_xticklabels(steps, rotation=20)
ax.set(title="Waterfall Chart", ylabel="cumulative value")
save(fig, "41_waterfall")

saved ./images\41_waterfall.png


'./images\\41_waterfall.png'

## Funnel Chart

Bordered trapezoids stacked vertically, narrower for each successive stage. Shows **drop-off** through a pipeline (sales funnel, signup funnel). Pitfall: doesn't measure time, only counts.

In [43]:
stages=["Visited","Signed up","Activated","Subscribed","Renewed"]
vals=[10000,4200,2100,950,720]
fig, ax = plt.subplots(figsize=(8,6))
maxw=1
for i,(s,v) in enumerate(zip(stages,vals)):
    w=v/vals[0]
    y=len(vals)-i-1
    ax.fill_betweenx([y-0.4, y+0.4], 0.5-w/2, 0.5+w/2, alpha=0.7,
                    color=plt.cm.viridis(i/len(vals)))
    ax.text(0.5, y, f"{s} — {v:,}", ha="center", color="white", fontweight="bold")
ax.set_xlim(0,1); ax.set_ylim(-0.6, len(vals)-0.4)
ax.set_xticks([]); ax.set_yticks([])
ax.set(title="Funnel Chart (conversion pipeline)")
save(fig, "42_funnel")

saved ./images\42_funnel.png


'./images\\42_funnel.png'

## Gantt Chart

Horizontal bars on a time axis — one per task. The **schedule workhorse** for project management. Pitfall: tiny tasks are invisible; consider Log time or zoom-in.

In [44]:
import datetime as dt
tasks=["Design","Build","Test","Review","Deploy","Docs"]
starts=[dt.date(2024,3,1),dt.date(2024,3,8),dt.date(2024,3,18),
        dt.date(2024,3,25),dt.date(2024,4,1),dt.date(2024,3,22)]
durs=[7,10,7,5,6,12]
fig, ax = plt.subplots(figsize=(9,5))
for i,(s,d) in enumerate(zip(starts,durs)):
    ax.barh(i, d, left=mdates.date2num(s), height=0.5,
            color=plt.cm.tab10(i))
    ax.text(mdates.date2num(s)+d/2, i, tasks[i], ha="center", va="center", color="white")
ax.set_yticks(range(len(tasks))); ax.set_yticklabels(tasks)
ax.xaxis.set_major_locator(mdates.DayLocator(interval=7))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.set(title="Gantt Chart"); plt.setp(ax.get_xticklabels(), rotation=30)
save(fig, "43_gantt")

saved ./images\43_gantt.png


'./images\\43_gantt.png'

## Sankey Diagram (matplotlib)

Flows shown as arrows whose **width is proportional** to the quantity moving from source to target. Use for **flows** (energy, money, migration).

In [45]:
from matplotlib.sankey import Sankey
fig=plt.figure(figsize=(8,6))
ax=fig.add_subplot(1,1,1)
s=Sankey(ax=ax, unit="TWh", scale=0.01)
s.add(flows=[100,30,-50,-80], labels=["Input","","Process A","Process B"],
       orientations=[0,-1,1,1])
s.finish()
ax.set_title("Sankey Diagram")
save(fig, "44_sankey")

Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


saved ./images\44_sankey.png


'./images\\44_sankey.png'

## Network Graph

Nodes (entities) and edges (relationships) laid out with a force simulation. Reads **structure of a graph** — communities, hubs, bridges. Pitfall: layout is heuristic; identical graphs can look different between runs.

In [46]:
G=nx.karate_club_graph()
pos=nx.spring_layout(G, seed=7)
degs=dict(G.degree())
fig, ax = plt.subplots(figsize=(8,7))
nx.draw_networkx_edges(G, pos, alpha=0.3, ax=ax)
nx.draw_networkx_nodes(G, pos, node_size=[v*40 for v in degs.values()],
                       node_color=list(degs.values()), cmap="plasma", ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
ax.set(title="Network Graph (Zachary karate club)"); ax.axis("off")
save(fig, "45_network")

saved ./images\45_network.png


'./images\\45_network.png'

## Dendrogram

Tree from hierarchical clustering — shows how observations merge into clusters. Cut the tree at a height to get a clustering. Best read as a **hierarchy**, not single-link distances.

In [47]:
from scipy.cluster.hierarchy import linkage, dendrogram
X=RNG.normal(0,1,(20,3))
Z=linkage(X, method="ward")
fig, ax = plt.subplots(figsize=(9,5))
dendrogram(Z, ax=ax, color_threshold=0.7*max(Z[:,2]))
ax.set(title="Dendrogram (ward linkage)")
save(fig, "46_dendrogram")

saved ./images\46_dendrogram.png


'./images\\46_dendrogram.png'

## Confusion Matrix Heatmap

Tabulates predicted vs actual classes from a classifier. Diagonal = correct, off-diagonal = errors. The **standard summary of classification performance**.

In [48]:
y_true=RNG.integers(0,5,400)
y_pred=y_true.copy()
for _ in range(60):
    i=RNG.integers(0,400); y_pred[i]=RNG.integers(0,5)
cm=confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6.5,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set(title="Confusion Matrix", xlabel="predicted", ylabel="actual")
save(fig, "47_confusion_matrix")

saved ./images\47_confusion_matrix.png


'./images\\47_confusion_matrix.png'

## ROC Curve

TPR vs FPR as the decision threshold sweeps. The diagonal is chance; the area (AUC) summarizes quality. **Threshold-free classifier comparison**.

In [49]:
from sklearn.metrics import roc_curve, auc
n=600
y=RNG.integers(0,2,n)
s=y*0.6 + RNG.normal(0,0.35,n)
y = np.clip(s + RNG.normal(0,0.1,n) > 0.5, 0, 1).astype(int)
# generate a plausible score
score = y + RNG.normal(0,0.5,n)
from sklearn.metrics import roc_auc_score
fpr,tpr,_=roc_curve(y, score)
fig, ax = plt.subplots(figsize=(6,6))
ax.plot(fpr,tpr,label=f"AUC={auc(fpr,tpr):.2f}", color="darkorange")
ax.plot([0,1],[0,1],"k--",alpha=0.5)
ax.set(title="ROC Curve", xlabel="false positive rate", ylabel="true positive rate")
ax.legend(loc="lower right")
save(fig, "48_roc")

saved ./images\48_roc.png


'./images\\48_roc.png'

## Q-Q Plot

Quantiles of the sample vs quantiles of a theoretical distribution (usually normal). Straight line = the data follows the distribution. The fastest way to **judge normality visually**.

In [50]:
data=RNG.normal(0,1,300)**2  # non-normal (chi-square with 1 df)
fig, ax = plt.subplots(figsize=(6.5,6.5))
stats.probplot(data, dist="norm", plot=ax)
ax.set_title("Q-Q Plot vs normal (curved ⇒ non-normal)")
save(fig, "49_qq")

saved ./images\49_qq.png


'./images\\49_qq.png'

## Autocorrelation Plot

Correlation of a series with a lagged copy of itself. Spikes at lag k mean the value at time t still carries information about time t−k. Standard preview before fitting any time-series model.

In [51]:
from matplotlib import pyplot as mp
x=np.cumsum(RNG.normal(0,1,500))
fig, ax = plt.subplots(figsize=(9,5))
ax.acorr(x, maxlags=24, normed=True)
ax.set(title="Autocorrelation Plot", xlabel="lag", xlim=(-25,25))
save(fig, "50_autocorrelation")

saved ./images\50_autocorrelation.png


'./images\\50_autocorrelation.png'

## Calendar Heatmap

Days on the calendar coloured by a value — daily counts of activity, Git commits, solar measurements. Lets you **see weekly and seasonal structure** at a glance.

In [52]:
days=pd.date_range("2024-01-01","2024-12-31", freq="D")
vals=RNG.integers(0,30,len(days)).astype(float)
vals = vals + (days.dayofweek.values<5)*8  # weekdays busier
# Build a 53-week x 7-dow grid of values
grid=np.full((53,7), np.nan)
for i,d in enumerate(days):
    iso=d.isocalendar()
    grid[iso.week-1, iso.weekday-1]=vals[i]
fig, ax = plt.subplots(figsize=(12,4))
im=ax.imshow(grid.T, aspect="auto", cmap="Greens", origin="upper")
ax.set_title("Calendar Heatmap (2024)")
ax.set_yticks(range(7)); ax.set_yticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
ax.set_xlabel("ISO week"); plt.colorbar(im, ax=ax, label="count")
save(fig, "51_calendar_heatmap")

saved ./images\51_calendar_heatmap.png


'./images\\51_calendar_heatmap.png'

## Plotly Interactive Line (exported PNG)

Plotly makes truly interactive charts (hover, zoom), useful in browser exploration. We export a static PNG so the gallery review still works without the JS runtime.

In [53]:
df=pd.DataFrame({
    "t": pd.date_range("2024-01-01", periods=60),
    "y": np.cumsum(RNG.normal(0,1,60))})
fig = px.line(df, x="t", y="y", title="Plotly Line")
save_plotly(fig, "52_plotly_line")     # uses plotly + kaleido

saved ./images\52_plotly_line.png


'./images\\52_plotly_line.png'

## Plotly Scatter with OLS Trendline

Plotly Express can add an OLS trendline in one keyword. Here we plot two noisy variables and the fitted line; residuals are visible from the scatter around the line.

In [54]:
n=160
df=pd.DataFrame({"x":RNG.normal(0,1,n)})
df["y"]=2.5*df["x"] + RNG.normal(0,1.5,n)
fig=px.scatter(df, x="x", y="y", trendline="ols", title="Plotly Scatter + OLS")
save_plotly(fig, "53_plotly_scatter_ols")

saved ./images\53_plotly_scatter_ols.png


'./images\\53_plotly_scatter_ols.png'

## Plotly 3D Surface

Plotly's WebGL surface renders smooth and is rotatable in the browser. Same surface as the matplotlib example, much smoother.

In [55]:
x=np.linspace(-3,3,60); y=np.linspace(-3,3,60)
X,Y=np.meshgrid(x,y); Z=np.sin(X)*np.cos(Y)*np.exp(-(X**2+Y**2)/10)
fig=go.Figure(data=[go.Surface(z=Z, x=x, y=y, colorscale="Viridis")])
fig.update_layout(title="Plotly 3D Surface", scene=dict(aspectratio=dict(x=1,y=1,z=0.6)))
save_plotly(fig, "54_plotly_surface3d")

saved ./images\54_plotly_surface3d.png


'./images\\54_plotly_surface3d.png'

## Plotly Treemap

Plotly's native treemap with **squarified** layout — better proportions than the matplotlib hand-rolled version. Hierarchical area chart.

In [56]:
df=pd.DataFrame({
    "name":[f"group {i}" for i in range(1,9)],
    "value": [40,28,22,18,12,9,7,4]})
fig=px.treemap(df, path=["name"], values="value", title="Plotly Treemap")
save_plotly(fig, "55_plotly_treemap")

saved ./images\55_plotly_treemap.png


'./images\\55_plotly_treemap.png'

## Plotly Sunburst

A radial hierarchy — concentric rings descend levels. Reads like a treemap bent into a circle. Great for **nested categories**.

In [57]:
df=pd.DataFrame({
    "parent": ["","","","A","A","B","B","C","C"],
    "label":  ["A","B","C","a1","a2","b1","b2","c1","c2"],
    "value":  [0,0,0, 25,15, 20,18, 12,10]})
fig=px.sunburst(df, names="label", parents="parent", values="value",
                title="Plotly Sunburst")
save_plotly(fig, "56_plotly_sunburst")

saved ./images\56_plotly_sunburst.png


'./images\\56_plotly_sunburst.png'

## Parallel Coordinates

Each data point is a polyline crossing one vertical axis per variable. The cleanest way to see **how one observation behaves across many dimensions** at once.

In [58]:
df=pd.DataFrame(RNG.normal(0,1,(150,5)), columns=list("ABCDE"))
df["cluster"]=KMeans(n_clusters=3, random_state=0, n_init=10).fit_predict(df)
fig=px.parallel_coordinates(df, color="cluster", dimensions=list("ABCDE"),
                            color_continuous_scale="viridis", title="Parallel Coordinates")
save_plotly(fig, "57_parallel_coords")

saved ./images\57_parallel_coords.png


'./images\\57_parallel_coords.png'

## Plotly Gapminder-style Bubble

Scatter with x, y, size = population, colour = continent — the framing Hans Rosling made famous. A single frame already tells a story; Plotly can animate over time.

In [59]:
df=pd.DataFrame({
    "gdp": RNG.uniform(1,5,40),
    "life": RNG.uniform(50,85,40),
    "pop": RNG.uniform(1,140,40),
    "cont": RNG.choice(["Asia","Europe","Africa","Americas"], 40)})
fig=px.scatter(df, x="gdp", y="life", size="pop", color="cont",
               log_x=True, size_max=40, title="Plotly Bubble (gapminder-style)")
save_plotly(fig, "58_plotly_bubble")

saved ./images\58_plotly_bubble.png


'./images\\58_plotly_bubble.png'

## Andrews Curve

Each row of a multivariate dataset becomes a single Fourier curve; similar rows make similar curves. A way to **see classes in high-D** without choosing two axes.

In [60]:
df=pd.DataFrame(RNG.normal(0,1,(150,4)), columns=list("ABCD"))
df["c"]=RNG.choice(["x","y","z"],150)
fig, ax = plt.subplots(figsize=(9,5))
pd.plotting.andrews_curves(df, "c", ax=ax, colormap="tab10")
ax.set(title="Andrews Curves")
save(fig, "59_andrews")

saved ./images\59_andrews.png


'./images\\59_andrews.png'

## Parallel Coordinates (pandas)

Pandas-native version of parallel coordinates — simpler but very useful as a one-liner for multi-class comparison across columns.

In [61]:
df=pd.DataFrame(RNG.normal(0,1,(120,5)), columns=list("ABCDE"))
df["label"]=RNG.choice(["x","y","z","w"],120)
fig, ax = plt.subplots(figsize=(9,5))
pd.plotting.parallel_coordinates(df, "label", ax=ax, colormap="tab10")
ax.set(title="Parallel Coordinates (pandas)")
save(fig, "60_parallel_coords_pandas")

saved ./images\60_parallel_coords_pandas.png


'./images\\60_parallel_coords_pandas.png'

## Lag Plot

Scatter of x(t) vs x(t+k). A tight diagonal means the series depends on its own past (autocorrelation). A shapeless blob means independence. Quick **serial-dependence check**.

In [62]:
x=np.cumsum(RNG.normal(0,1,400))
fig, ax = plt.subplots(figsize=(6,6))
pd.plotting.lag_plot(pd.Series(x), lag=1, ax=ax)
ax.set(title="Lag Plot (lag = 1)")
save(fig, "61_lag_plot")

saved ./images\61_lag_plot.png


'./images\\61_lag_plot.png'

## RadViz

Multivariate plotting where each variable is an anchor on a circle and points are pulled toward anchors by their value. A different take on **class separation in many dimensions**.

In [63]:
df=pd.DataFrame(RNG.normal(0,1,(150,5)), columns=list("ABCDE"))
df["label"]=RNG.choice(["x","y","z"],150)
fig, ax = plt.subplots(figsize=(7,7))
pd.plotting.radviz(df, "label", ax=ax, colormap="tab10")
ax.set_title("RadViz")
save(fig, "62_radviz")

saved ./images\62_radviz.png


'./images\\62_radviz.png'

## Bootstrap CI Visualization

A histogram of bootstrap-resampled means with the 95 % CI overlaid. Shows where the true mean probably lives — **sampling-distribution thinking** in one picture.

In [64]:
sample=RNG.normal(5,2,60)
means=[RNG.choice(sample,60,replace=True).mean() for _ in range(3000)]
lo,hi=np.percentile(means,[2.5,97.5])
fig, ax = plt.subplots(figsize=(8,5))
ax.hist(means, bins=40, color="steelblue", edgecolor="white")
ax.axvline(lo, color="crimson", linestyle="--", label=f"2.5% = {lo:.2f}")
ax.axvline(hi, color="crimson", linestyle="--", label=f"97.5% = {hi:.2f}")
ax.set(title="Bootstrap mean — 95% CI"); ax.legend()
save(fig, "63_bootstrap")

saved ./images\63_bootstrap.png


'./images\\63_bootstrap.png'

## Clustering Comparison Plot

Same data coloured by **true label** (left) vs **k-means prediction** (right). A direct way to show a model's success (and failure modes).

In [65]:
X,y=make_blobs(n_samples=300, centers=3, random_state=4)
pred=KMeans(n_clusters=3, random_state=0, n_init=10).fit_predict(X)
fig, axes = plt.subplots(1,2, figsize=(12,5))
for ax,lab,title in [(axes[0],y,"true"),(axes[1],pred,"k-means predicted")]:
    ax.scatter(X[:,0],X[:,1], c=lab, cmap="tab10", s=20)
    ax.set_title(title); ax.set_xlabel("x1"); ax.set_ylabel("x2")
fig.suptitle("Clustering Comparison", y=1.02)
save(fig, "64_clustering")

saved ./images\64_clustering.png


'./images\\64_clustering.png'

## Decision Regions — 2D Classifier

Mesh the feature space with a fine grid, classify each cell, colour the background. Shows the **full decision boundary** of a 2-D classifier at a glance.

In [66]:
from matplotlib.colors import ListedColormap
X,y=make_moons(n_samples=200, noise=0.2, random_state=0)
clf=KMeans(n_clusters=2, random_state=0, n_init=10).fit(X)
xx=np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 250)
yy=np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 250)
g1,g2=np.meshgrid(xx,yy)
Z=clf.predict(np.c_[g1.ravel(), g2.ravel()]).reshape(g1.shape)
fig, ax = plt.subplots(figsize=(7,5.5))
ax.contourf(g1,g2,Z, alpha=0.25, cmap=ListedColormap(["skyblue","salmon"]))
ax.scatter(X[:,0],X[:,1], c=y, cmap=ListedColormap(["navy","darkred"]), s=20, edgecolor="white")
ax.set(title="Decision Regions (k-means on moons)", xlabel="x1", ylabel="x2")
save(fig, "65_decision_regions")

saved ./images\65_decision_regions.png


'./images\\65_decision_regions.png'

## Word Frequency Bars

Without the `wordcloud` package, the honest way to show word counts is a horizontal bar chart sorted by frequency. More readable than a wordcloud and free of font-weight distortions.

In [67]:
import collections
sample="the the cat sat on the mat the dog ran on the path the sun rose over the hill on the path the dog ran the dog ran the".split()
wc=collections.Counter(sample).most_common(12)
words,cnts=zip(*wc)
fig, ax = plt.subplots(figsize=(8,5))
ax.barh(words[::-1], cnts[::-1], color="purple")
ax.set(title="Word Frequency Bars", xlabel="count")
save(fig, "66_wordfreq")

saved ./images\66_wordfreq.png


'./images\\66_wordfreq.png'

## Spiral / Cycle Plot

A continuous series wrapped into a spiral whose angle is the phase of a cycle (e.g. day-of-year) and radius is time elapsed. Surfaces **periodic structure + long-term trend** together. Asteroseismology and solar physics use these heavily.

In [68]:
n=365*4
t=np.arange(n)
y=np.sin(2*np.pi*t/365) + 0.3*np.sin(2*np.pi*t/27) + RNG.normal(0,0.2,n)
theta=2*np.pi*(t%365)/365
r=t/365
fig, ax = plt.subplots(figsize=(7,7), subplot_kw=dict(projection="polar"))
sc=ax.scatter(theta, r, c=y, cmap="coolwarm", s=8)
plt.colorbar(sc, ax=ax, label="value"); ax.set(title="Spiral / Cycle Plot (4 years)")
save(fig, "67_spiral")

saved ./images\67_spiral.png


'./images\\67_spiral.png'

## Spectrogram

A signal split into windows and its frequency content at each window stacked vertically. Shows **how the spectrum evolves in time** — speech, music, helioseismology.

In [69]:
fs=1000; t=np.arange(0,4,1/fs); n=len(t)
x=np.concatenate([np.sin(2*np.pi*50*t[:n//2]),
                   np.sin(2*np.pi*120*t[n//2:])]) + 0.2*RNG.normal(0,1,n)
fig, ax = plt.subplots(figsize=(9,5))
P, freqs, bins, im = ax.specgram(x, NFFT=256, Fs=fs, noverlap=128, cmap="magma")
plt.colorbar(im, ax=ax, label="power")
ax.set(title="Spectrogram", xlabel="time", ylabel="frequency (Hz)")
save(fig, "68_spectrogram")

saved ./images\68_spectrogram.png


'./images\\68_spectrogram.png'

## Manhattan-style Peak Plot

A scatter sorted by position where each spike is a 'hit.' Originally from GWAS, useful anywhere you want to see whether a mostly-flat sequence hides a few big peaks.

In [70]:
pos=np.arange(500)
heights=np.abs(RNG.normal(0,1,500))
heights[40]+=6; heights[180]+=4.5; heights[420]+=5
fig, ax = plt.subplots(figsize=(9,5))
ax.scatter(pos, heights, c=heights, cmap="viridis", s=12)
ax.axhline(3, color="red", linestyle="--", label="significance")
ax.set(title="Manhattan-style Peak Plot", xlabel="position", ylabel="signal")
ax.legend()
save(fig, "69_manhattan")

saved ./images\69_manhattan.png


'./images\\69_manhattan.png'

## Empirical CDF (ECDF)

Cumulative fraction of the data ≤ x. Reads like a stairs-to-smooth curve of the distribution. Robust to binning (unlike a histogram), and comparing distributions reduces to comparing curves.

In [71]:
a=RNG.normal(0,1,300); b=RNG.normal(1,1.2,300)
fig, ax = plt.subplots(figsize=(8,5))
sns.ecdfplot(a, ax=ax, label="A"); sns.ecdfplot(b, ax=ax, label="B")
ax.set(title="Empirical CDF"); ax.legend()
save(fig, "70_ecdf")

saved ./images\70_ecdf.png


'./images\\70_ecdf.png'

## Strip Chart with Jitter

A 1-D scatter jittered along y to spread points horizontally. Use to look at the **raw distribution of a single group** or to overlay the actual points on a boxplot.

In [72]:
g1=RNG.normal(0,1,80); g2=RNG.normal(2,1.2,80)
fig, ax = plt.subplots(figsize=(8,4))
ax.scatter(np.zeros_like(g1)+RNG.uniform(-0.05,0.05,80), g1, alpha=0.6, label="A")
ax.scatter(np.ones_like(g2)+1+RNG.uniform(-0.05,0.05,80), g2, alpha=0.6, label="B")
ax.set_xticks([0,1]); ax.set_xticklabels(["A","B"])
ax.set(title="Strip Chart with Jitter"); ax.legend()
save(fig, "71_strip_jitter")

saved ./images\71_strip_jitter.png


'./images\\71_strip_jitter.png'

## Lorenz Curve

Cumulative share of the variable vs cumulative share of the population (sorted). Distance from the diagonal = inequality; the area is the **Gini coefficient**. Standard for income/wealth distribution.

In [73]:
x=RNG.lognormal(0,1,500); x.sort()
ls=np.arange(1,len(x)+1)/len(x)
ys=np.cumsum(x)/x.sum()
lo=np.concatenate([[0], ls]); yo=np.concatenate([[0], ys])
fig, ax = plt.subplots(figsize=(6.5,6.5))
ax.plot(lo, yo, label="data"); ax.plot([0,1],[0,1], "--k", label="equality")
ax.fill_between(lo, yo, lo, alpha=0.2)
ax.set(title="Lorenz Curve", xlabel="cumulative population",
       ylabel="cumulative share"); ax.legend()
save(fig, "72_lorenz")

saved ./images\72_lorenz.png


'./images\\72_lorenz.png'

## MDS Embedding (2D)

Project a high-D distance matrix to 2-D so distances are preserved as well as possible. Use to **see cluster structure** when PCA's linear projection masks non-linear shape.

In [74]:
X,y=make_circles(n_samples=300, noise=0.08, factor=0.5, random_state=0)
emb=MDS(n_components=2, random_state=0, normalized_stress="auto").fit_transform(X)
fig, ax = plt.subplots(figsize=(6.5,6.5))
ax.scatter(emb[:,0],emb[:,1], c=y, cmap="tab10", s=20)
ax.set(title="MDS Embedding (2D)"); ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
save(fig, "73_mds")

saved ./images\73_mds.png


'./images\\73_mds.png'

## PCA Explained Variance

Bar of how much variance each principal component captures. The **detour or sharp drop** marks how many PCs you need to keep.

In [75]:
X,y=make_blobs(n_samples=300, centers=4, n_features=8, random_state=0)
pca=PCA().fit(X)
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(range(1,9), pca.explained_variance_ratio_, color="mediumpurple")
ax.plot(range(1,9), np.cumsum(pca.explained_variance_ratio_), "o-", color="darkred")
ax.set(title="PCA — explained variance per PC", xlabel="PC",
       ylabel="fraction (bars) / cumulative (line)")
save(fig, "74_pca_variance")

saved ./images\74_pca_variance.png


'./images\\74_pca_variance.png'

## Seaborn Clustermap (heatmap + dendrograms)

Heatmap with hierarchical clustering dendrograms on both axes. Great when you don't know the **natural groupings** in a matrix — the algorithm orders rows and columns so similar entries are adjacent.

In [76]:
mat=RNG.normal(0,1,(30,12))
mat[:15] += 1; mat[15:22] -= 1
df=pd.DataFrame(mat, index=[f"r{i}" for i in range(30)],
                columns=[f"c{i}" for i in range(12)])
g=sns.clustermap(df, cmap="vlag", center=0, figsize=(9,10),
                 row_cluster=True, col_cluster=True)
g.fig.suptitle("Clustermap", y=1.01)
fig = g.fig
save(fig, "75_clustermap")

saved ./images\75_clustermap.png


'./images\\75_clustermap.png'

## Residuals vs Fitted

After fitting a regression, plotting residuals against the fitted values is the most informative single **diagnostic**. Pattern in this plot = the model is missing structure (non-linearity, heteroscedasticity).

In [77]:
import numpy as np
x=RNG.uniform(0,10,200); y=1 + 0.8*x + RNG.normal(0,1.5,200)
fit=np.polyfit(x,y,1); pred=np.polyval(fit,x); resid=y-pred
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(pred, resid, alpha=0.6, edgecolor="none")
ax.axhline(0, color="k", linestyle="--")
ax.set(title="Residuals vs Fitted", xlabel="fitted", ylabel="residual")
save(fig, "76_residuals")

saved ./images\76_residuals.png


'./images\\76_residuals.png'

## Sparklines (small multiples)

Tiny line charts that fit inline next to a label. Use for **dense comparison of trends** across many series; signal conveyed at a glance without labeled axes.

In [78]:
rows=6
fig, axes = plt.subplots(rows, 1, figsize=(8, rows*0.7), sharex=True)
for i,ax in enumerate(axes):
    y=np.cumsum(RNG.normal(0,1,100))
    ax.plot(y, color=plt.cm.tab10(i), linewidth=1.5)
    ax.set_xticks([]); ax.set_yticks([])
    ax.spines[["top","right","left","bottom"]].set_visible(False)
    ax.set_ylabel(f"s{i}", rotation=0, labelpad=15, va="center")
axes[-1].set_xlabel("time")
fig.suptitle("Sparklines (small-multiple trends)", y=0.95)
save(fig, "77_sparkline")

saved ./images\77_sparkline.png


'./images\\77_sparkline.png'

## Line with Confidence Band

A central line plus a shaded band (e.g., CI, min/max, or ±σ). Use when each y has an **associated range**, not just a single value.

In [79]:
x=np.linspace(0,10,100); y=np.sin(x); band=RNG.uniform(0.1,0.3,100)
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(x, y, color="darkblue")
ax.fill_between(x, y-band, y+band, color="darkblue", alpha=0.2)
ax.set(title="Line with Confidence Band", xlabel="x", ylabel="y")
save(fig, "78_band")

saved ./images\78_band.png


'./images\\78_band.png'

## Grouped KDE comparison

Several KDEs from different groups on the same axes — like overlapping smoothed histograms. Best for **comparing distributions of one variable across categories**.

In [80]:
df=pd.DataFrame({
    "x": np.concatenate([RNG.normal(0,1,200), RNG.normal(2,1,200),
                         RNG.normal(-1,1.5,200)]),
    "g": ["A"]*200 + ["B"]*200 + ["C"]*200})
fig, ax = plt.subplots(figsize=(8,5))
sns.kdeplot(data=df, x="x", hue="g", fill=True, alpha=0.35, ax=ax,
            common_norm=False)
ax.set(title="Grouped KDE comparison")
save(fig, "79_grouped_kde")

saved ./images\79_grouped_kde.png


'./images\\79_grouped_kde.png'

## Cumulative Sum (Running Total)

Step chart of ∑xᵢ against i. Essential for **time-series that are intrinsically accumulative** (case counts, balances, radioactivity).

In [81]:
x=RNG.normal(0,1,200)
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(np.cumsum(x), color="indigo")
ax.set(title="Cumulative Sum", ylabel="running total", xlabel="step")
save(fig, "80_cumsum")

saved ./images\80_cumsum.png


'./images\\80_cumsum.png'

## Two-tone Bar (Pareto-ish)

A bar chart where bars above a threshold are coloured differently. Common in **Pareto analysis** — highlight the few categories that drive most of the total.

In [82]:
cats=list("ABCDEFGHIJ"); vals=np.sort(RNG.integers(1,40,10))[::-1]
thr=20
colors=["crimson" if v>=thr else "steelblue" for v in vals]
fig, ax = plt.subplots(figsize=(9,5))
ax.bar(cats, vals, color=colors)
ax.axhline(thr, color="k", linestyle="--", label=f"threshold={thr}")
ax.set(title="Two-tone Bar (Pareto highlight)", ylabel="value"); ax.legend()
save(fig, "81_pareto")

saved ./images\81_pareto.png


'./images\\81_pareto.png'

## Poisson vs observed bar overlay

Bars = observed frequencies of an integer count, markers = the model Poisson probabilities. The standard hand-check for a count model fit.

In [83]:
lam=3
obs=RNG.poisson(lam, 1000)
xs=np.arange(0, max(obs)+1)
freq=np.bincount(obs, minlength=xs.size)/len(obs)
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(xs, freq, color="lightblue", label="observed")
ax.plot(xs, stats.poisson.pmf(xs, lam), "o-", color="darkred", label=f"Poisson(λ={lam})")
ax.set(title="Observed vs Poisson model"); ax.legend()
save(fig, "82_poisson_overlay")

saved ./images\82_poisson_overlay.png


'./images\\82_poisson_overlay.png'

## Image Display (imshow)

Render a 2-D array as an image — pixel = colour of one cell. Used for raw array data (satellite tiles, MRI), for adding spatial context to matrices. Always choose a perceptually uniform colormap unless you have a reason not to.

In [84]:
im=RNG.uniform(0,1,(64,64))
# Running integral → smooth field for nicer display
im=np.cumsum(np.cumsum(im,axis=0),axis=1)
im=(im-np.min(im))/(np.max(im)-np.min(im))
fig, ax = plt.subplots(figsize=(6.5,6))
im=ax.imshow(im, cmap="magma", interpolation="bilinear")
plt.colorbar(im, ax=ax); ax.set_title("Image Display (imshow)")
save(fig, "83_imshow")

saved ./images\83_imshow.png


'./images\\83_imshow.png'

## Image + Contour overlay

imshow + iso-lines on top. Useful when you want both the **continuous field** and discrete levels — satellite cloud images, model fields.

In [85]:
x=np.linspace(-3,3,150); y=np.linspace(-3,3,150)
X,Y=np.meshgrid(x,y); Z=np.sin(X)*np.cos(Y)*np.exp(-(X**2+Y**2)/10)
fig, ax = plt.subplots(figsize=(7,6))
im=ax.imshow(Z, extent=[-3,3,-3,3], origin="lower", cmap="viridis")
ax.contour(X,Y,Z, levels=8, colors="white", linewidths=0.5)
ax.set(title="Image + Contour overlay"); plt.colorbar(im, ax=ax)
save(fig, "84_img_contour")

saved ./images\84_img_contour.png


'./images\\84_img_contour.png'

## Twin-Y Axes Plot

Two series sharing an x-axis but with **different units** (e.g. temperature in °C and humidity in %). Use rarely and color both axes to match their series — otherwise it's misleading.

In [86]:
x=np.linspace(0,10,100); y1=np.sin(x)+20; y2=50*np.exp(-x/3)
fig, ax = plt.subplots(figsize=(9,5))
l1,=ax.plot(x, y1, color="darkred"); ax.set_ylabel("temp. (°C)", color="darkred")
ax2=ax.twinx()
l2,=ax2.plot(x, y2, color="navy"); ax2.set_ylabel("pressure (kPa)", color="navy")
ax.set(title="Twin-Y Axes Plot", xlabel="x")
ax.legend([l1,l2],["temp","pressure"], loc="upper right")
save(fig, "85_twin_y")

saved ./images\85_twin_y.png


'./images\\85_twin_y.png'

## Broken-Y Axis Plot

Bars with one big value that would crush the others — a **y-axis break** lets you zoom both the low group and the high outlier. Use sparingly; it can mislead readers who miss the zigzag marker.

In [87]:
from matplotlib.transforms import Bbox
cats=list("ABCDE"); vals=[8,12,9,11,220]
fig, (ax1,ax2)=plt.subplots(2,1, sharex=True, figsize=(8,5),
                            gridspec_kw=dict(height_ratios=[1,2]))
for ax in (ax1, ax2):
    ax.bar(cats, vals, color="teal")
ax1.set_ylim(150,240); ax2.set_ylim(0,20)
ax2.spines["top"].set_visible(False); ax1.spines["bottom"].set_visible(False)
ax1.tick_params(bottom=False)
d=0.008
ax1.plot((-d,d),(-d/2,d/2), transform=ax1.transAxes, clip_on=False, color="k")
ax2.plot((-d,d),(1-d/2,1+d/2), transform=ax2.transAxes, clip_on=False, color="k")
fig.suptitle("Broken-Y Axis (one outlier)")
save(fig, "86_broken_axis")

saved ./images\86_broken_axis.png


'./images\\86_broken_axis.png'

## Log-Log Scatter

Both axes logarithmic. Straight line ⇒ power law relationship, with slope = exponent. The go-to plot for **power-law detection** across many orders of magnitude.

In [88]:
n=200; x=RNG.uniform(0.1,1000,n); y=2.5*x**1.4*RNG.uniform(0.5,1.5,n)
fig, ax = plt.subplots(figsize=(8,6))
ax.loglog(x, y, 'o', alpha=0.5, color="darkorange")
ax.set(title="Log-Log Scatter", xlabel="x (log)", ylabel="y (log)")
save(fig, "87_loglog")

saved ./images\87_loglog.png


'./images\\87_loglog.png'

## Semi-Log Y Plot

Log y, linear x. Standard for plotting **things spanning many orders** (radioactive decay, epidemic counts) where a straight line ⇒ exponential.

In [89]:
x=np.linspace(0,10,100); y=1000*np.exp(-0.6*x)
fig, ax = plt.subplots(figsize=(8,5))
ax.semilogy(x, y, 'o-', color="indigo")
ax.set(title="Semi-Log Y Plot (exponential decay)", xlabel="x", ylabel="count (log)")
save(fig, "88_semilog")

saved ./images\88_semilog.png


'./images\\88_semilog.png'

## Bland-Altman (Tukey mean-difference) Plot

Difference between two measurements vs their mean. Standard for agreement between two instruments/methods. Limits = ±1.96·SD of the differences (= 95 % agreement range).

In [90]:
m1=RNG.normal(100,8,80)+RNG.normal(0,3,80)  # method 1
m2=m1+RNG.normal(0,4,80)                        # method 2
avg=(m1+m2)/2; diff=m1-m2
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(avg, diff, alpha=0.7)
ax.axhline(diff.mean(), color="darkblue", label="mean")
ax.axhline(diff.mean()+1.96*diff.std(), color="crimson", linestyle="--", label="+1.96 SD")
ax.axhline(diff.mean()-1.96*diff.std(), color="crimson", linestyle="--", label="-1.96 SD")
ax.set(title="Bland-Altman Plot", xlabel="mean of methods", ylabel="difference")
ax.legend()
save(fig, "89_bland_altman")

saved ./images\89_bland_altman.png


'./images\\89_bland_altman.png'

## Violin + Box Overlay

Combines the shape of a violin (distribution) with the summary stats of a boxplot. Useful when the violin's smooth shape alone hides the quartiles.

In [91]:
df=pd.DataFrame({"v": np.concatenate(
    [RNG.normal(0,1,300), RNG.normal(2,1.3,300), RNG.normal(5,0.9,300)]),
    "g": ["A"]*300+["B"]*300+["C"]*300})
fig, ax = plt.subplots(figsize=(8,5))
sns.violinplot(data=df, x="g", y="v", inner="box", ax=ax)
ax.set(title="Violin + box overlay")
save(fig, "90_violin_overlay")

saved ./images\90_violin_overlay.png


'./images\\90_violin_overlay.png'

## Done — package everything for download

The next cell zips every PNG in `./images/` into a single archive, "
`./visualizations_gallery.zip`, so you can grab the whole gallery in one "
download.

In [92]:
from pathlib import Path
images = sorted(Path(IMG).glob("*.png"))
zip_name = "visualizations_gallery.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in images:
        zf.write(p, arcname=p.name)
print(f"Wrote {zip_name} with {len(images)} images")

Wrote visualizations_gallery.zip with 90 images


## Quick index of the gallery

Below is a compact list of every chart generated, with a one-line purpose.
Each line ties the chart name to the saved PNG in `./images/`.

01 `01_line_chart.png` — basic 2-D line for a continuous variable.
02 `02_multiline_timeseries.png` — several series on a shared time axis.
03 `03_area_chart.png` — filled area under a line.
04 `04_stacked_area.png` — layers stacked to show composition over time.
05 `05_bar_chart.png` — categorical magnitudes as bars.
06 `06_grouped_bar.png` — two series side-by-side per category.
07 `07_stacked_bar.png` — segments stacked within bars.
08 `08_horizontal_bar.png` — bars run L-R for long labels/ranking.
09 `09_histogram.png` — distribution of a single continuous variable.
10 `10_density_kde.png` — smoothed density estimate.
11 `11_box_plot.png` — median/quartiles/outliers per group.
12 `12_violin_plot.png` — box plot + KDE.
13 `13_swarm_plot.png` — every observation as a non-overlapping point.
14 `14_scatter.png` — classic bivariate scatter.
15 `15_bubble_chart.png` — scatter + colour + size.
16 `16_hexbin.png` — 2-D density via hexagonal bins.
17 `17_kde2d.png` — smooth contours of bivariate density.
18 `18_pie_chart.png` — parts of a whole; use sparingly.
19 `19_donut.png` — pie with hole — slightly more legible.
20 `20_heatmap.png` — matrix as colour.
21 `21_correlation_heatmap.png` — pairwise correlation matrix.
22 `22_pairplot.png` — every pairwise scatter.
23 `23_jointplot.png` — scatter + marginal histograms.
24 `24_errorbar.png` — line chart with uncertainty bars.
25 `25_boxen.png` — box plot with extra tail quantiles.
26 `26_ridge.png` — overlapping KDEs stacked vertically.
27 `27_stem.png` — lollipop/stem for discrete sequences.
28 `28_step.png` — staircase for piecewise-constant data.
29 `29_candlestick.png` — OHLC for price history.
30 `30_radar.png` — multi-axis comparison of one entity.
31 `31_polar_bar.png` — bars on a circle for cyclic data.
32 `32_polar_line.png` — line in polar coordinates.
33 `33_contour.png` — iso-lines of a 2-D field.
34 `34_quiver.png` — vector field as arrows.
35 `35_streamplot.png` — continuous streamlines of a flow.
36 `36_surface3d.png` — 2-D function rendered as a 3-D sheet.
37 `37_wireframe.png` — 3-D mesh.
38 `38_scatter3d.png` — three continuous variables in 3-D.
39 `39_treemap.png` — area = quantity.
40 `40_waffle.png` — one square per unit.
41 `41_waterfall.png` — cumulative increments to a total.
42 `42_funnel.png` — drop-off through a pipeline.
43 `43_gantt.png` — schedule with bars on a time axis.
44 `44_sankey.png` — flows whose width encodes quantity.
45 `45_network.png` — graph of nodes + edges.
46 `46_dendrogram.png` — hierarchical clustering tree.
47 `47_confusion_matrix.png` — predicted vs actual heatmap.
48 `48_roc.png` — TPR vs FPR with AUC.
49 `49_qq.png` — sample vs normal quantiles.
50 `50_autocorrelation.png` — series vs lagged self.
51 `51_calendar_heatmap.png` — days of year coloured by value.
52 `52_plotly_line.png` — Plotly interactive line (exported PNG).
53 `53_plotly_scatter_ols.png` — Plotly scatter with OLS trendline.
54 `54_plotly_surface3d.png` — Plotly WebGL surface.
55 `55_plotly_treemap.png` — Plotly squarified treemap.
56 `56_plotly_sunburst.png` — Plotly radial hierarchy.
57 `57_parallel_coords.png` — plotly parallel coordinates.
58 `58_plotly_bubble.png` — Plotly gapminder-style bubble.
59 `59_andrews.png` — row-wise Fourier curves.
60 `60_parallel_coords_pandas.png` — pandas parallel coordinates.
61 `61_lag_plot.png` — lag-1 scatter for serial dependence.
62 `62_radviz.png` — radial-anchor multivariate plot.
63 `63_bootstrap.png` — bootstrap distribution of the mean.
64 `64_clustering.png` — true vs k-means labels side by side.
65 `65_decision_regions.png` — background-coloured classifier regions.
66 `66_wordfreq.png` — word frequency bar chart.
67 `67_spiral.png` — cycle phase + radius wrap of a series.
68 `68_spectrogram.png` — time-frequency waterfall.
69 `69_manhattan.png` — peaks over a mostly flat sequence.
70 `70_ecdf.png` — empirical CDF.
71 `71_strip_jitter.png` — jittered 1-D scatter per group.
72 `72_lorenz.png` — inequality curve and equality diagonal.
73 `73_mds.png` — MDS 2-D embedding of a distance matrix.
74 `74_pca_variance.png` — variance explained per PC.
75 `75_clustermap.png` — heatmap with row/column dendrograms.
76 `76_residuals.png` — regression residuals vs fitted.
77 `77_sparkline.png` — small-multiple trend lines.
78 `78_band.png` — line with confidence band.
79 `79_grouped_kde.png` — several KDEs across groups.
80 `80_cumsum.png` — running total.
81 `81_pareto.png` — bars with a significance threshold.
82 `82_poisson_overlay.png` — observed counts vs Poisson model.
83 `83_imshow.png` — display a 2-D array as an image.
84 `84_img_contour.png` — image with iso-lines on top.
85 `85_twin_y.png` — two series sharing x, dual y scales.
86 `86_broken_axis.png` — zigzag y for one outlier.
87 `87_loglog.png` — log-log scatter (power-law search).
88 `88_semilog.png` — semi-log y for exponential signal.
89 `89_bland_altman.png` — method-agreement mean-difference.
90 `90_violin_overlay.png` — violin with embedded box.

That is **90** chart types plus the index zip.